# 卒論：改善版特徴量の再解析・段位別比較・固定モデル評価

**研究対象は「しょぼんｎ」本人の低段位期・高段位期の打ち方の違いです。**
旧23特徴量・H1〜H6・旧成果物は保持し、新しい出力先へ保存します。

- 段位16以下／18以上の定義を維持。段位17と初期1500観測対局は履歴に含め、採点から除外します。
- 主方式は `hour_invariant`。`order_verified=False`、`full_history_confirmed=False` を維持します。
- **このNotebookの配布時点では、全対局の改善版統計・実データ学習・外部性能は未実行です。**
- 記述分析を主目的とし、固定LightGBMは補助評価です。ウルトラ立直の結果に基づく選択・調整をしません。

**C1→C2→C3→C4を実行して件数・試験抽出を確認し、その後にC5以降を実行してください。**
初回から「すべてのセルを実行」は使わず、段階ごとに確認します。
C5とC8は不足分の牌譜解析、C7は研究用モデル学習、C9は外部採点を行います。
C10はC6直後にも実行でき、未実行の評価を数値で埋めません。

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## C1：固定コード・依存関係・Drive接続・実装テスト

初回にColabの読み込み済みライブラリと固定版が異なる場合、導入後にPythonセッションを一度再起動します。
**接続が戻ったらC1をもう一度実行してください。** その後、Drive接続と実装テストへ進みます。
再起動によりセッション内の変数が消えるため、C1は分析セルより先に実行してください。Drive上の保存データは保持されます。
テストには合成データを使用します。


In [1]:
from pathlib import Path
import hashlib, importlib.metadata, json, os, re, subprocess, sys, urllib.request

CODE_COMMIT = "15572a5a3f21da470c4fcbb0e16496f31cba0b3e"  # 配布時に40桁の固定コミットへ置換
if not re.fullmatch(r"[0-9a-f]{40}", CODE_COMMIT):
    raise RuntimeError("固定コードのコミット未設定です。公開された完成版Notebookを開いてください。")
CODE_ROOT = Path("/content/mahjong_research_v2") / CODE_COMMIT
CODE_ROOT.mkdir(parents=True, exist_ok=True)
for name, module in tuple(sys.modules.items()):
    if name == "mahjong_research" or name.startswith("mahjong_research."):
        loaded_path = getattr(module, "__file__", None)
        if loaded_path and not Path(loaded_path).resolve().is_relative_to(CODE_ROOT.resolve()):
            raise RuntimeError("別コミットの研究コードが読込済みです。Colabセッションを再起動してC1を再実行してください。")
SOURCE_URL = f"https://raw.githubusercontent.com/yuukienomoto/report_-syobon_n/{CODE_COMMIT}/research_v2/"
FILES = [
    "requirements.txt", "README.md", "legacy_feature_audit.csv",
    "feature_definitions_v2.csv", "old_to_v2_mapping.csv",
    "mahjong_research/__init__.py", "mahjong_research/legacy.py",
    "mahjong_research/tenhou.py", "mahjong_research/inputs.py",
    "mahjong_research/analysis.py", "mahjong_research/reporting.py",
    "mahjong_research/workflow.py", "tests/test_tenhou.py",
    "mahjong_research/recovery.py", "tests/test_disclosure_regression.py",
    "tests/fixtures/disclosed_own_four_tiles.xml",
    "tests/test_inputs.py", "tests/test_analysis.py", "tests/test_workflow.py",
    "docs/feature_audit_ja.md", "docs/thesis_draft_ja.md",
    "docs/extractor_validation.json", "docs/extractor_validation.md",
    "docs/analysis_validation.json", "docs/analysis_validation_ja.md",
]
manifest_path = CODE_ROOT / "bootstrap_manifest.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
downloaded = 0
for relative in FILES:
    dest = CODE_ROOT / relative
    if dest.is_file() and manifest.get(relative) == hashlib.sha256(dest.read_bytes()).hexdigest():
        continue
    with urllib.request.urlopen(SOURCE_URL + relative, timeout=60) as response:
        payload = response.read()
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(payload)
    manifest[relative] = hashlib.sha256(payload).hexdigest()
    downloaded += 1
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

expected = {}
for line in (CODE_ROOT / "requirements.txt").read_text().splitlines():
    if "==" in line and not line.strip().startswith("#"):
        package, version = line.strip().split("==", 1)
        expected[package] = version
module_names = {"scikit-learn": "sklearn"}
loaded_mismatch = []
need_install = False
for package, version in expected.items():
    module = sys.modules.get(module_names.get(package, package))
    if module is not None and getattr(module, "__version__", None) != version:
        loaded_mismatch.append(package)
    try:
        need_install |= importlib.metadata.version(package) != version
    except importlib.metadata.PackageNotFoundError:
        need_install = True
if need_install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_ROOT / "requirements.txt")])
if loaded_mismatch:
    # Native libraries already imported by Colab need a fresh Python process.
    # A persistent guard prevents an automatic restart loop.
    restart_guard = CODE_ROOT / "dependency_restart.json"
    previous = json.loads(restart_guard.read_text(encoding="utf-8")) if restart_guard.exists() else None
    if previous and previous.get("expected") == expected and not need_install:
        raise SystemExit("再起動後もライブラリの版が一致しません。［ランタイム］からセッションを再起動してC1を再実行してください。改善しない場合はC1の出力を共有してください。対象: " + ", ".join(loaded_mismatch))
    restart_guard.write_text(json.dumps({"expected": expected, "pid": os.getpid()}, indent=2), encoding="utf-8")
    print("依存関係の準備が完了しました。読み込み済みライブラリを切り替えるため、Pythonセッションを一度再起動します。", flush=True)
    print("接続が戻ったらC1をもう一度実行してください。次にテストとDrive接続へ進みます。", flush=True)
    os.kill(os.getpid(), 9)  # SIGKILL on the Linux Colab runtime
    raise SystemExit("セッションの再起動を待ち、C1を再実行してください。")
for package, version in expected.items():
    if importlib.metadata.version(package) != version:
        raise RuntimeError(f"依存関係が一致しません: {package}=={version}")

os.environ["MPLCONFIGDIR"] = str(CODE_ROOT / ".matplotlib")
sys.path.insert(0, str(CODE_ROOT)) if str(CODE_ROOT) not in sys.path else None
from google.colab import drive
drive.mount("/content/drive")
import pandas as pd
from IPython.display import display, Markdown
from mahjong_research.analysis import AnalysisBlocked, load_frozen_bundle
from mahjong_research.workflow import (
    initialize, preview, pilot, extract_stage, descriptive_stage,
    model_stage, external_stage, final_report,
)
import unittest
suite = unittest.defaultTestLoader.discover(str(CODE_ROOT / "tests"), pattern="test_*.py")
test_result = unittest.TextTestRunner(verbosity=1).run(suite)
if not test_result.wasSuccessful():
    raise RuntimeError("実装テストが失敗しました。実データ解析へ進まないでください。")
print("固定コード:", CODE_COMMIT, "今回の取得:", downloaded, "ファイル")
print("実装テスト完了。実データの全件抽出・モデル学習は未実行です。")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


..........................

shobon: 再利用 1 / 今回解析 1 / 全対象候補 2
保存済み 2 / 2
test: 再利用 0 / 今回解析 1 / 全対象候補 1
保存済み 1 / 1
test: 完了済みキャッシュを検証・再利用 / 今回解析 0


....

test: 再利用 0 / 今回解析 1 / 全対象候補 1
保存済み 1 / 1
test: 完了済みキャッシュを検証・再利用 / 今回解析 0
test: 再利用 0 / 今回解析 1 / 全対象候補 1
保存済み 1 / 1
test: 完了済みキャッシュを検証・再利用 / 今回解析 0
test: 再利用 0 / 今回解析 1 / 全対象候補 1
保存済み 1 / 1
test: 完了済みキャッシュを検証・再利用 / 今回解析 0
test: 再利用 0 / 今回解析 1 / 全対象候補 1
保存済み 1 / 1
test: 再利用 0 / 今回解析 3 / 全対象候補 3
保存済み 3 / 3


...

test: 再利用 0 / 今回解析 2 / 全対象候補 3
保存済み 1 / 3
保存済み 2 / 3
試験抽出のみ。全件完了ではありません。
test: 再利用 2 / 今回解析 1 / 全対象候補 3
保存済み 3 / 3
test: 完了済みキャッシュを検証・再利用 / 今回解析 0
test: 再利用 0 / 今回解析 0 / 全対象候補 1
試験抽出のみ。全件完了ではありません。
test: 再利用 0 / 今回解析 1 / 全対象候補 1
保存済み 1 / 1


............................
----------------------------------------------------------------------
Ran 61 tests in 16.367s

OK


固定コード: 15572a5a3f21da470c4fcbb0e16496f31cba0b3e 今回の取得: 0 ファイル
実装テスト完了。実データの全件抽出・モデル学習は未実行です。


## C2：入力パス・新しい実行フォルダ

ZIP名が異なる場合だけ実際のパスに変更します。旧キャッシュ候補が複数ある場合は
`LEGACY_DIRS` に `raw_manifest.csv` と `raw_features.csv` のあるフォルダを指定します。
再開時は前回表示された `RUN_DIR` を `RESUME_RUN_DIR` に指定してください。
保存済み旧CSVを新版として読み替えることはしません。

In [2]:
ROOT = Path("/content/drive/MyDrive/卒論")
ZIP_PATHS = {
    "shobon": ROOT / "mjlog_pf4-20_n30.zip",
    "ultra": ROOT / "ultrareach.zip",
}
LEGACY_DIRS = {
    "shobon": None,  # 例: ROOT / "validation_inputs/zip_cache/shobon_旧署名"
    "ultra": None,
}
RESUME_RUN_DIR = None  # 再開時だけ前回RUN_DIRのパス文字列を指定
ctx = initialize(ROOT, CODE_ROOT, CODE_COMMIT, ZIP_PATHS, LEGACY_DIRS, resume=RESUME_RUN_DIR)
RUN_DIR = Path(ctx["folder"])
print("RUN_DIR =", RUN_DIR)
print("order_verified=False / full_history_confirmed=False")

# Reset in-memory results so another run cannot leak into this report.
old_desc = new_desc = legacy_bundle = new_bundle = external_metrics = None
old = plans = shared = new_shobon = new_ultra = None


RUN_DIR = /content/drive/MyDrive/卒論/research_v2_runs/20261105T071722Z_5da6c9ad
order_verified=False / full_history_confirmed=False


## C3：入力監査・共通対局・処理件数の事前表示

旧CSV・キャッシュの署名とZIP一覧を照合します。牌譜の全件再解析は行いません。
不足ファイルや未解決要素が表示された場合は、その内容を確認してください。
共通対局を含む窓は学習から除外し、履歴の行を詰め直しません。

In [3]:
old, plans, shared = preview(ctx)
display(pd.DataFrame([
    {"player": key, "旧特徴量対局数": len(old[key]),
     "抽出対象数": len(plan["members"]), "新版キャッシュ済み": plan["already_cached"],
     "未抽出数": plan["remaining"], "旧アーカイブ未解決要素": plan["legacy_unresolved_entries"],
     "新版キャッシュ保存先": plan["cache_dir"]}
    for key, plan in plans.items()
]))
print("両プレイヤーに共通する対局ID:", len(shared), "件")


,player,旧特徴量対局数,抽出対象数,新版キャッシュ済み,未抽出数,旧アーカイブ未解決要素,新版キャッシュ保存先
0,shobon,11573,11573,11573,0,0,/content/drive/MyDrive/卒論/research_v2_cache/sh...
1,ultra,6053,6053,0,6053,6711,/content/drive/MyDrive/卒論/research_v2_cache/ul...


両プレイヤーに共通する対局ID: 26 件


## C3R：今回の1対局エラーから復旧する場合だけ実行
C1・C2・C3の後で実行します。C2は新しい実行フォルダを作成してください。旧抽出器と全バッチ・入力の署名が一致する場合だけ正常行を新しいキャッシュへ移行します。旧キャッシュは保持されます。表示が「正常11572、残り1」ならC5へ進んでください。通常の初回解析ではこのセルを飛ばします。


In [4]:
from mahjong_research.recovery import recover_successful_batches
OLD_SHOBON_CACHE = ROOT / "research_v2_cache/shobon_4105129b8f8dbace9d04"
plans["shobon"] = recover_successful_batches(plans["shobon"], OLD_SHOBON_CACHE)
print("署名照合済みの正常キャッシュ:", plans["shobon"]["already_cached"])
print("再解析する対象数:", plans["shobon"]["remaining"])
print("新しい保存先:", plans["shobon"]["cache_dir"])


署名照合済みの正常キャッシュ: 11573
再解析する対象数: 0
新しい保存先: /content/drive/MyDrive/卒論/research_v2_cache/shobon_fcaa4c7bbf0d28bc8252


## C4：しょぼんｎの少数牌譜で試験抽出

未抽出から最大8件を処理します。既存の新版キャッシュは再利用します。
手牌・イベント整合性の失敗があれば、その対局を黙って削除せず停止します。
この試験が成功しても、全件の正しさを確認したことにはなりません。

In [ ]:
pilot_audit = pilot(ctx, plans["shobon"], max_new=8)
display(pilot_audit)
print("少数確認の範囲です。全件の品質確認はC5で行います。")


## C5：しょぼんｎの新版特徴量を全件生成・検査

**このセルで未抽出分の牌譜解析を実行します。** C3/C4の結果を確認してから進んでください。
途中で切断されても、同じコード・入力の署名付きバッチキャッシュを再利用できます。
初期期間・段位17を含む全対局を保持し、補完前・移動平均前の新版特徴量と分子分母を保存します。

In [5]:
new_shobon, quality_shobon = extract_stage(ctx, plans["shobon"])
display(quality_shobon)
print("しょぼんｎ 新版対局数:", len(new_shobon))


shobon: 完了済みキャッシュを検証・再利用 / 今回解析 0


,feature,n,missing,missing_rate,minimum,maximum,infinite,out_of_range,invalid_ratio,no_opportunity
0,first_response_genbutsu_rate,11573,71,0.006135,0.0,1.000000,0,0,0,71
1,response_non_genbutsu_rate,11573,71,0.006135,0.0,1.000000,0,0,0,71
2,response_suji_nongenbutsu_rate,11573,71,0.006135,0.0,1.000000,0,0,0,71
3,response_nonsuji_suited_rate,11573,71,0.006135,0.0,1.000000,0,0,0,71
4,early_noten_non_genbutsu_rate,11573,3957,0.341917,0.0,1.000000,0,0,0,3957
5,mid_noten_non_genbutsu_rate,11573,443,0.038279,0.0,1.000000,0,0,0,443
6,late_noten_non_genbutsu_rate,11573,2055,0.177568,0.0,1.000000,0,0,0,2055
7,roundstart_top_non_genbutsu_rate,11573,6248,0.539877,0.0,1.000000,0,0,0,6248
8,roundstart_middle_non_genbutsu_rate,11573,3285,0.283850,0.0,1.000000,0,0,0,3285
9,roundstart_last_non_genbutsu_rate,11573,7069,0.610818,0.0,1.000000,0,0,0,7069


しょぼんｎ 新版対局数: 11573


## C6：旧版・新版の段位別比較、分布、月別推移

同じ終端・元対局集合の `hour_invariant` 窓で比較します。
**記述分析は補完せず、特徴量ごとに10対局すべてに値がある窓だけ平均を定義します。**
希少行動では有効窓が不足する場合があるため、欠損率、対局単位、非重複窓、機会数も確認します。
ここでの分布はH4/H5の補完済み入力分布と一致するとは限りません。
重複窓を独立標本とみなした検定は行いません。

In [6]:
old_desc, new_desc, descriptive_out = descriptive_stage(
    ctx, old["shobon"], new_shobon, shared, make_plots=True
)
display(pd.DataFrame([{"version": "legacy23", **old_desc["summary"]},
                      {"version": "v2", **new_desc["summary"]}]))
display(new_desc["differences"].query("view == 'primary_hour_invariant'")
        .sort_values("abs_smd", ascending=False))
print("比較図表・日本語レポート:", descriptive_out)
print("記述分析のみで終える場合も、C10を実行して実行状態を記録できます。")


,version,player,observed_games,cutoff_hour,initially_excluded,primary_windows,disjoint_windows,unique_source_games,max_source_reuse,order_verified,full_history_confirmed,method,missing_policy,overlapping_windows_are_independent,plots_generated
0,legacy23,しょぼんｎ,11573,2012-06-23 03:00:00,1501,3215,547,6014,10,False,False,hour_invariant,no imputation; per-feature all 10 values required,False,True
1,v2,しょぼんｎ,11573,2012-06-23 03:00:00,1501,3215,547,6014,10,False,False,hour_invariant,no imputation; per-feature all 10 values required,False,True


,view,feature,mean_diff_high_minus_low,pooled_sd,smd,abs_smd,low_valid_n,high_valid_n
44,primary_hour_invariant,early_noten_non_genbutsu_rate,0.069503,0.104772,0.663373,0.663373,65,16
40,primary_hour_invariant,first_response_genbutsu_rate,-0.020029,0.077140,-0.259651,0.259651,2355,740
43,primary_hour_invariant,response_nonsuji_suited_rate,0.011603,0.047553,0.244000,0.244000,2355,740
59,primary_hour_invariant,exhaustive_draw_disclosed_tenpai_rate,-0.029130,0.125523,-0.232067,0.232067,267,72
56,primary_hour_invariant,early_first_open_meld_rate,0.009561,0.043877,0.217897,0.217897,2463,752
46,primary_hour_invariant,late_noten_non_genbutsu_rate,0.014656,0.070098,0.209073,0.209073,410,104
41,primary_hour_invariant,response_non_genbutsu_rate,0.008395,0.047620,0.176284,0.176284,2355,740
45,primary_hour_invariant,mid_noten_non_genbutsu_rate,0.009858,0.066636,0.147930,0.147930,1724,557
51,primary_hour_invariant,chasing_riichi_share,0.012106,0.088357,0.137012,0.137012,551,218
55,primary_hour_invariant,open_meld_round_rate,0.005761,0.053613,0.107459,0.107459,2463,752


比較図表・日本語レポート: /content/drive/MyDrive/卒論/research_v2_runs/20261105T071722Z_5da6c9ad/descriptive/20261105T071841Z_6dd8fea0
記述分析のみで終える場合も、C10を実行して実行状態を記録できます。


## C6B：分子・分母を合算する追加検証
C6後に実行します。このセルだけを現在実行中のNotebookへコピーして使えます。新しいセッションではC1後にC6_REFERENCE_DIRへ前回のC6保存先を指定すれば、C2〜C6を再実行せず解析できます。キャッシュから読み取り、追加結果は卒論/research_v2_opportunity/<新実行ID>へ保存します。
18割合と2イベント加重平均を、従来方式・同じ元対局非重複窓・共通有効窓で比較します。改善項目数と差の安定性はこのセルの実データ出力で確認してください。C7へ進む前にreport_ja.mdと欠損・機会数を確認してください。


In [7]:
# C6B: run after C6, or set the saved C6 directory below after C1.
from pathlib import Path
import hashlib, importlib.util, json, sys, urllib.request
import pandas as pd
from IPython.display import display, Markdown

SUPPLEMENT_COMMIT = "6c6b9f46ae39c615c38409e3d9e36f37297a9909"
C6_REFERENCE_DIR = None  # e.g. "/content/drive/MyDrive/卒論/research_v2_runs/.../descriptive/..."
if C6_REFERENCE_DIR is None:
    C6_REFERENCE_DIR = globals().get("descriptive_out")
if C6_REFERENCE_DIR is None:
    raise ValueError("C6_REFERENCE_DIRに前回C6が表示した保存先（descriptive/<実行ID>）を設定してください。")
reference = Path(C6_REFERENCE_DIR)
run_dir = reference.parent.parent
reference_meta_path = reference / "v2" / "primary_windows.csv"
for required in [reference_meta_path, run_dir/"input_plan.json", run_dir/"shared_games.csv"]:
    if not required.is_file():
        raise FileNotFoundError(f"C6参照ファイルがありません: {required}")
run_config = json.loads((run_dir/"run_configuration.json").read_text(encoding="utf-8"))
if run_config["code_commit"] != CODE_COMMIT:
    raise ValueError("C6の固定コードとC1のCODE_COMMITが違います。元のC6と同じコードを使用してください。")
cache_dir = Path(json.loads((run_dir/"input_plan.json").read_text(encoding="utf-8"))["shobon"]["cache_dir"])
raw_path = cache_dir / "raw_features_v2.csv"
complete = json.loads((cache_dir/"complete.json").read_text(encoding="utf-8"))
from mahjong_research import tenhou as extractor
if (hashlib.sha256(raw_path.read_bytes()).hexdigest() != complete["sha256"]
        or hashlib.sha256(Path(extractor.__file__).read_bytes()).hexdigest() != complete["extractor_sha256"]
        or complete.get("order_verified") is not False or complete.get("full_history_confirmed") is not False):
    raise ValueError("C6キャッシュのハッシュ・抽出器・確認フラグの照合に失敗しました。")
data_for_supplement = pd.read_csv(raw_path, float_precision="round_trip")
if len(data_for_supplement) != complete["rows"]:
    raise ValueError("C6キャッシュの対局件数が一致しません。")
reference_meta = pd.read_csv(reference_meta_path)
shared_for_supplement = set(pd.read_csv(run_dir/"shared_games.csv").game_id)
supplement_root = Path("/content/mahjong_opportunity_supplement") / SUPPLEMENT_COMMIT
supplement_root.mkdir(parents=True, exist_ok=True)
for filename in ["opportunity_analysis.py", "test_opportunity_analysis.py"]:
    url=f"https://raw.githubusercontent.com/yuukienomoto/report_-syobon_n/{SUPPLEMENT_COMMIT}/research_v2/supplement/{filename}"
    payload=urllib.request.urlopen(url, timeout=60).read()
    (supplement_root/filename).write_bytes(payload)
module_name = "opportunity_analysis"
spec=importlib.util.spec_from_file_location(module_name, supplement_root/"opportunity_analysis.py")
module=importlib.util.module_from_spec(spec)
sys.modules[module_name]=module
spec.loader.exec_module(module)
import unittest
test_spec=importlib.util.spec_from_file_location("test_opportunity_analysis",supplement_root/"test_opportunity_analysis.py")
test_module=importlib.util.module_from_spec(test_spec)
test_spec.loader.exec_module(test_module)
test_result=unittest.TextTestRunner(verbosity=1).run(unittest.defaultTestLoader.loadTestsFromModule(test_module))
if not test_result.wasSuccessful():
    raise RuntimeError("追加集計のテストに失敗しました。入力を変更せず停止します。")
drive_root=Path(run_config["root"])
opportunity_result=module.run_opportunity_analysis(data_for_supplement,shared_for_supplement,reference_meta,
                                                  drive_root/"research_v2_opportunity")
display(pd.DataFrame([opportunity_result["facts"]]))
display(opportunity_result["comparisons"])
display(Markdown((opportunity_result["output_dir"]/"report_ja.md").read_text(encoding="utf-8")))
print("追加分析の保存先:", opportunity_result["output_dir"])


......
----------------------------------------------------------------------
Ran 6 tests in 3.766s

OK


,primary_windows,disjoint_windows,unique_source_games,max_source_reuse,features_improved_either_group,features_improved_both_groups,stable_direction_candidates,order_verified,full_history_confirmed,imputation,data_sha256
0,3215,547,6014,10,16,16,"[first_response_genbutsu_rate, response_non_ge...",False,False,none,2b5b928de51bf9a04eac84f73a812bd16cea532fbcf264...


,feature,traditional_overlap_mean_diff_high_minus_low,traditional_overlap_smd,traditional_overlap_low_valid_n,traditional_overlap_high_valid_n,opportunity_overlap_mean_diff_high_minus_low,opportunity_overlap_smd,opportunity_overlap_low_valid_n,opportunity_overlap_high_valid_n,traditional_disjoint_mean_diff_high_minus_low,...,opportunity_common_overlap_low_valid_n,opportunity_common_overlap_high_valid_n,traditional_common_disjoint_mean_diff_high_minus_low,traditional_common_disjoint_smd,traditional_common_disjoint_low_valid_n,traditional_common_disjoint_high_valid_n,opportunity_common_disjoint_mean_diff_high_minus_low,opportunity_common_disjoint_smd,opportunity_common_disjoint_low_valid_n,opportunity_common_disjoint_high_valid_n
0,first_response_genbutsu_rate,-0.020029,-0.259651,2355,740,-0.019544,-0.286068,2463,752,-0.020692,...,2355,740,-0.020692,-0.267486,410,114,-0.019004,-0.275820,410,114
1,response_non_genbutsu_rate,0.008395,0.176284,2355,740,0.005900,0.151145,2463,752,0.009103,...,2355,740,0.009103,0.192646,410,114,0.006048,0.156512,410,114
2,response_suji_nongenbutsu_rate,0.000603,0.020633,2355,740,0.004171,0.166931,2463,752,0.002211,...,2355,740,0.002211,0.075190,410,114,0.005298,0.206099,410,114
3,response_nonsuji_suited_rate,0.011603,0.244000,2355,740,0.003059,0.079140,2463,752,0.009740,...,2355,740,0.009740,0.195791,410,114,0.001927,0.047910,410,114
4,early_noten_non_genbutsu_rate,0.069503,0.663373,65,16,0.008401,0.064856,2463,752,0.017714,...,65,16,0.017714,0.133661,5,2,0.051857,0.503544,5,2
5,mid_noten_non_genbutsu_rate,0.009858,0.147930,1724,557,0.009893,0.175749,2463,752,0.006618,...,1724,557,0.006618,0.101206,295,84,0.009214,0.166361,295,84
6,late_noten_non_genbutsu_rate,0.014656,0.209073,410,104,-0.001158,-0.016739,2463,752,-0.009173,...,410,104,-0.009173,-0.133836,76,16,0.021778,0.424567,76,16
7,roundstart_top_non_genbutsu_rate,NaN,NaN,2,0,0.006190,0.068435,2460,752,NaN,...,2,0,NaN,NaN,0,0,NaN,NaN,0,0
8,roundstart_middle_non_genbutsu_rate,-0.004964,-0.065990,86,30,0.008155,0.122823,2463,752,-0.046845,...,86,30,-0.046845,-0.462361,11,4,-0.007906,-0.116177,11,4
9,roundstart_last_non_genbutsu_rate,NaN,NaN,0,0,0.011265,0.105835,2437,747,NaN,...,0,0,NaN,NaN,0,0,NaN,NaN,0,0


# C6追加検証：機会数に基づく10対局集計

主分析3215窓、元対局非重複547窓。最大再利用10回。
有効窓数が少なくとも一方の群で増えた指標は16/20、両群で増えた指標は16/20。

## 各特徴量の確認結果
- 成立リーチごとの初回応答現物率：有効窓数 低2355→2463、高740→752。高−低の平均差 -0.02003→-0.01954。集計方式間は同方向、追加方式の重複/非重複間は同方向。
- 対リーチ応答非現物率：有効窓数 低2355→2463、高740→752。高−低の平均差 0.008395→0.0059。集計方式間は同方向、追加方式の重複/非重複間は同方向。
- 対リーチ応答非現物スジ率：有効窓数 低2355→2463、高740→752。高−低の平均差 0.0006026→0.004171。集計方式間は同方向、追加方式の重複/非重複間は同方向。
- 対リーチ応答非現物非スジ数牌率：有効窓数 低2355→2463、高740→752。高−低の平均差 0.0116→0.003059。集計方式間は同方向、追加方式の重複/非重複間は同方向。
- 自己ツモ6回以下・打牌後形状ノーテン非現物率：有効窓数 低65→2463、高16→752。高−低の平均差 0.0695→0.008401。集計方式間は同方向、追加方式の重複/非重複間は同方向。
- 自己ツモ7～12回・打牌後形状ノーテン非現物率：有効窓数 低1724→2463、高557→752。高−低の平均差 0.009858→0.009893。集計方式間は同方向、追加方式の重複/非重複間は同方向。
- 自己ツモ13回以上・打牌後形状ノーテン非現物率：有効窓数 低410→2463、高104→752。高−低の平均差 0.01466→-0.001158。集計方式間は逆方向、追加方式の重複/非重複間は逆方向。
- 局開始時単独1位の非現物応答率：有効窓数 低2→2460、高0→752。高−低の平均差 nan→0.00619。集計方式間は比較不能、追加方式の重複/非重複間は同方向。
- 局開始時同点なし2・3位の非現物応答率：有効窓数 低86→2463、高30→752。高−低の平均差 -0.004964→0.008155。集計方式間は逆方向、追加方式の重複/非重複間は同方向。
- 局開始時単独4位の非現物応答率：有効窓数 低0→2437、高0→747。高−低の平均差 nan→0.01126。集計方式間は比較不能、追加方式の重複/非重複間は同方向。
- 成立リーチ時の表ドラ枚数平均：有効窓数 低551→2463、高218→752。高−低の平均差 -0.0009249→-0.003299。集計方式間は同方向、追加方式の重複/非重複間は同方向。
- 自分の成立リーチに占める追っかけ割合：有効窓数 低551→2463、高218→752。高−低の平均差 0.01211→0.0001936。集計方式間は同方向、追加方式の重複/非重複間は同方向。
- 成立追っかけリーチの形状待ち種類数平均：有効窓数 低0→2302、高0→713。高−低の平均差 nan→-0.01352。集計方式間は比較不能、追加方式の重複/非重複間は同方向。
- 成立追っかけリーチの両面解釈可能率：有効窓数 低0→2302、高0→713。高−低の平均差 nan→-0.008581。集計方式間は比較不能、追加方式の重複/非重複間は同方向。
- 成立追っかけリーチの表ドラ0割合：有効窓数 低0→2302、高0→713。高−低の平均差 nan→0.01014。集計方式間は比較不能、追加方式の重複/非重複間は同方向。
- 1回以上副露した局の割合：有効窓数 低2463→2463、高752→752。高−低の平均差 0.005761→0.004974。集計方式間は同方向、追加方式の重複/非重複間は同方向。
- 自己ツモ6回以下の初副露局割合：有効窓数 低2463→2463、高752→752。高−低の平均差 0.009561→0.00891。集計方式間は同方向、追加方式の重複/非重複間は同方向。
- 自己ツモ13回以上の初副露局割合：有効窓数 低2463→2463、高752→752。高−低の平均差 -0.00129→-0.0004837。集計方式間は同方向、追加方式の重複/非重複間は同方向。
- 初副露成立直後に表ドラを持つ局割合：有効窓数 低2463→2463、高752→752。高−低の平均差 0.003632→0.002875。集計方式間は同方向、追加方式の重複/非重複間は同方向。
- 通常流局時の手牌開示率：有効窓数 低267→2463、高72→752。高−低の平均差 -0.02913→-0.01585。集計方式間は同方向、追加方式の重複/非重複間は同方向。

## 解釈と推奨
18項目は機会あたりの割合、表ドラ平均と待ち種類数平均の2項目はイベントあたりの数量の加重平均。いずれも分子・分母が保存済みなので再解析せず計算する。
機会がない対局の0/0は行動率0とは扱わない。窓内の機会数合計が0なら欠損。10対局窓全体のイベントに関する記述を主目的にする場合、機会数による集計を主分析の候補とし、対局率の単純平均は別の推定対象として感度分析に残す。
主な比較表では方式により有効窓の集合が変わる。共通有効窓のcommon分析を併記し、重みの変更と対象窓の変更を分けて確認する。
窓内ではイベント数を重みにするが、群の平均値は有効な10対局窓を等重みで平均する。期間全体の全イベントを一括合算した率とは推定対象が異なる。
同方向という表示は差の符号の一致だけを示す。分母が少ない窓・有効窓数・SMDの大きさも確認し、安定した実力差や統計的有意差と解釈しない。
窓間でも元対局が重複する。検定・独立標本に基づく信頼区間は算出していない。非重複窓も同一人物の時系列であり独立性は保証されない。
窓は終端段位で分類するので段位17や異なる段位の履歴を含む。時間経過、対戦相手、卓条件、発生機会の差が交絡し得る。
order_verified=False、full_history_confirmed=False。全プレイヤーへの一般化と因果解釈はできない。分類モデルの入力やパラメータは本追加分析で変更していない。

追加分析の保存先: /content/drive/MyDrive/卒論/research_v2_opportunity/20261105T071935Z_fe1c05dd


## C7：固定LightGBMの内部評価と外部評価前のモデル固定

**このセルで研究用モデルを学習します。** 固定200木・深さ5・学習率0.03等を使用し、探索や早期終了を行いません。
訓練区間だけで補完値を推定し、窓の元対局重複と共有対局混入を検査します。
学習が単一クラスの分割は停止・記録し、単一クラス評価のAUC／Balanced Accuracyは算出しません。
旧版の学習前提が満たされない場合は理由を保存し、新版の処理を別に試します。
同じ実行フォルダに固定モデルがあれば、検証して再利用します。

In [8]:
legacy_bundle = None
new_bundle = None
try:
    legacy_dir = RUN_DIR / "model_legacy"
    if legacy_dir.exists() and any(legacy_dir.iterdir()) and not (legacy_dir / "external_freeze.json").exists():
        raise AnalysisBlocked("Previous legacy model attempt is incomplete; preserve its audit and continue only v2.")
    legacy_bundle = model_stage(ctx, old["shobon"], shared, version="legacy")
    display(legacy_bundle["fold_audit"])
    display(legacy_bundle["metrics"])
except AnalysisBlocked as exc:
    from datetime import datetime, timezone
    import uuid
    status_dir = RUN_DIR / "status"
    status_dir.mkdir(exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
    status_path = status_dir / f"legacy_model_blocked_{stamp}.json"
    status_path.write_text(json.dumps({"status": "blocked", "reason": str(exc),
        "note": "旧版性能は未算出。数値を補わない。"}, ensure_ascii=False, indent=2), encoding="utf-8")
    print("旧版の学習を停止:", exc, "\n記録:", status_path)
new_bundle = model_stage(ctx, new_shobon, shared, version="v2")
display(new_bundle["fold_audit"])
display(new_bundle["metrics"])
print("外部評価前の固定モデル:", new_bundle["model_dir"])


,fold,train_last_hour,validation_first_hour,train_weak,train_strong,validation_weak,validation_strong,source_overlap,status
0,1,2021-11-13 12:00:00,2021-11-14 00:00:00,1015,242,489,0,0,evaluated_single_class_AUC_BA_NA
1,2,2023-07-22 16:00:00,2023-07-22 18:00:00,1508,242,292,189,0,evaluated
2,3,2024-12-05 21:00:00,2024-12-05 22:00:00,1800,431,657,321,0,evaluated


,dataset,estimator,n,weak,strong,strong_ratio,accuracy,balanced_accuracy,roc_auc,f1_strong,f1_macro,recall_weak,recall_strong,tn,fp,fn,tp
0,fold1,LightGBM,489,489,0,0.000000,0.877301,NaN,NaN,0.000000,0.467320,0.877301,NaN,429,60,0,0
1,fold1,always_weak,489,489,0,0.000000,1.000000,NaN,NaN,0.000000,0.500000,1.000000,NaN,489,0,0,0
2,fold1,always_strong,489,489,0,0.000000,0.000000,NaN,NaN,0.000000,0.000000,0.000000,NaN,0,489,0,0
3,fold1,training_majority,489,489,0,0.000000,1.000000,NaN,NaN,0.000000,0.500000,1.000000,NaN,489,0,0,0
4,fold2,LightGBM,481,292,189,0.392931,0.590437,0.506831,0.580615,0.182573,0.454670,0.897260,0.116402,262,30,167,22
5,fold2,always_weak,481,292,189,0.392931,0.607069,0.500000,0.500000,0.000000,0.377749,1.000000,0.000000,292,0,189,0
6,fold2,always_strong,481,292,189,0.392931,0.392931,0.500000,0.500000,0.564179,0.282090,0.000000,1.000000,0,292,0,189
7,fold2,training_majority,481,292,189,0.392931,0.607069,0.500000,0.500000,0.000000,0.377749,1.000000,0.000000,292,0,189,0
8,fold3,LightGBM,978,657,321,0.328221,0.529652,0.452368,0.454563,0.240924,0.450092,0.677321,0.227414,445,212,248,73
9,fold3,always_weak,978,657,321,0.328221,0.671779,0.500000,0.500000,0.000000,0.401835,1.000000,0.000000,657,0,321,0


,fold,train_last_hour,validation_first_hour,train_weak,train_strong,validation_weak,validation_strong,source_overlap,status
0,1,2021-11-13 12:00:00,2021-11-14 00:00:00,1015,242,489,0,0,evaluated_single_class_AUC_BA_NA
1,2,2023-07-22 16:00:00,2023-07-22 18:00:00,1508,242,292,189,0,evaluated
2,3,2024-12-05 21:00:00,2024-12-05 22:00:00,1800,431,657,321,0,evaluated


,dataset,estimator,n,weak,strong,strong_ratio,accuracy,balanced_accuracy,roc_auc,f1_strong,f1_macro,recall_weak,recall_strong,tn,fp,fn,tp
0,fold1,LightGBM,489,489,0,0.000000,0.905930,NaN,NaN,0.000000,0.475322,0.905930,NaN,443,46,0,0
1,fold1,always_weak,489,489,0,0.000000,1.000000,NaN,NaN,0.000000,0.500000,1.000000,NaN,489,0,0,0
2,fold1,always_strong,489,489,0,0.000000,0.000000,NaN,NaN,0.000000,0.000000,0.000000,NaN,0,489,0,0
3,fold1,training_majority,489,489,0,0.000000,1.000000,NaN,NaN,0.000000,0.500000,1.000000,NaN,489,0,0,0
4,fold2,LightGBM,481,292,189,0.392931,0.611227,0.520222,0.561372,0.161435,0.454195,0.945205,0.095238,276,16,171,18
5,fold2,always_weak,481,292,189,0.392931,0.607069,0.500000,0.500000,0.000000,0.377749,1.000000,0.000000,292,0,189,0
6,fold2,always_strong,481,292,189,0.392931,0.392931,0.500000,0.500000,0.564179,0.282090,0.000000,1.000000,0,292,0,189
7,fold2,training_majority,481,292,189,0.392931,0.607069,0.500000,0.500000,0.000000,0.377749,1.000000,0.000000,292,0,189,0
8,fold3,LightGBM,978,657,321,0.328221,0.492843,0.414615,0.424136,0.194805,0.412328,0.642314,0.186916,422,235,261,60
9,fold3,always_weak,978,657,321,0.328221,0.671779,0.500000,0.500000,0.000000,0.401835,1.000000,0.000000,657,0,321,0


外部評価前の固定モデル: /content/drive/MyDrive/卒論/research_v2_runs/20261105T071722Z_5da6c9ad/model_v2


## C8：ウルトラ立直の新版特徴量生成・検査

**このセルで外部プレイヤーの未抽出分を解析します。**
C7で固定した新版モデルの記録・前処理を検証してから処理を開始します。
外部プレイヤーの数値に基づく特徴量選択、パラメータ・閾値変更は行いません。

In [9]:
if globals().get("new_bundle") is None:
    raise RuntimeError("C7で新版モデルを固定・再読込してからC8を実行してください。")
new_bundle = load_frozen_bundle(new_bundle["model_dir"])
new_ultra, quality_ultra = extract_stage(ctx, plans["ultra"])
display(quality_ultra)
print("ウルトラ立直 新版対局数:", len(new_ultra))


ultra: 再利用 0 / 今回解析 6053 / 全対象候補 6053
保存済み 250 / 6053
保存済み 500 / 6053
保存済み 750 / 6053
保存済み 1000 / 6053
保存済み 1250 / 6053
保存済み 1500 / 6053
保存済み 1750 / 6053
保存済み 2000 / 6053
保存済み 2250 / 6053
保存済み 2500 / 6053
保存済み 2750 / 6053
保存済み 3000 / 6053
保存済み 3250 / 6053
保存済み 3500 / 6053
保存済み 3750 / 6053
保存済み 4000 / 6053
保存済み 4250 / 6053
保存済み 4500 / 6053
保存済み 4750 / 6053
保存済み 5000 / 6053
保存済み 5250 / 6053
保存済み 5500 / 6053
保存済み 5750 / 6053
保存済み 6000 / 6053
保存済み 6053 / 6053


,feature,n,missing,missing_rate,minimum,maximum,infinite,out_of_range,invalid_ratio,no_opportunity
0,first_response_genbutsu_rate,6053,72,0.011895,0.0,1.0,0,0,0,72
1,response_non_genbutsu_rate,6053,72,0.011895,0.0,1.0,0,0,0,72
2,response_suji_nongenbutsu_rate,6053,72,0.011895,0.0,1.0,0,0,0,72
3,response_nonsuji_suited_rate,6053,72,0.011895,0.0,1.0,0,0,0,72
4,early_noten_non_genbutsu_rate,6053,2038,0.336693,0.0,1.0,0,0,0,2038
5,mid_noten_non_genbutsu_rate,6053,276,0.045597,0.0,1.0,0,0,0,276
6,late_noten_non_genbutsu_rate,6053,1118,0.184702,0.0,1.0,0,0,0,1118
7,roundstart_top_non_genbutsu_rate,6053,3229,0.533454,0.0,1.0,0,0,0,3229
8,roundstart_middle_non_genbutsu_rate,6053,1862,0.307616,0.0,1.0,0,0,0,1862
9,roundstart_last_non_genbutsu_rate,6053,3890,0.642657,0.0,1.0,0,0,0,3890


ウルトラ立直 新版対局数: 6053


## C9：固定モデルの外部評価

固定モデル・訓練補完値で一度採点します。外部データでは学習しません。
完了済みの同一入力・モデル・出力先なら、保存済み結果だけを読み込みます。
途中で失敗した外部評価は黙ってやり直さず、記録を確認します。

In [10]:
external_metrics = external_stage(ctx, new_bundle, new_ultra)
display(external_metrics)
print("外部評価のクラス構成・混同行列・予測スコア分布も保存しました。")


,dataset,estimator,n,weak,strong,strong_ratio,accuracy,balanced_accuracy,roc_auc,f1_strong,f1_macro,recall_weak,recall_strong,tn,fp,fn,tp
0,external,LightGBM,713,97,616,0.863955,0.274895,0.354524,0.292743,0.368742,0.258506,0.463918,0.24513,45,52,465,151
1,external,always_weak,713,97,616,0.863955,0.136045,0.500000,0.500000,0.000000,0.119753,1.000000,0.00000,97,0,616,0
2,external,always_strong,713,97,616,0.863955,0.863955,0.500000,0.500000,0.927013,0.463506,0.000000,1.00000,0,97,0,616
3,external,training_majority,713,97,616,0.863955,0.136045,0.500000,0.500000,0.000000,0.119753,1.000000,0.00000,97,0,616,0


外部評価のクラス構成・混同行列・予測スコア分布も保存しました。


## C10：卒論用の性能比較表、日本語結果・考察、実行状態

C6の記述比較後から実行可能です。未実行のモデル・外部評価は未実行のまま記録します。
従来ランダム分割AUC0.8340、従来外部Accuracy35.59%、H5の既報値は、今回計算した数値と区別します。
平均差は同一人物の段位別の関連であり、実力向上の原因や強者全体の特徴とは断定しません。
C10はこの実行で作成した図を `figures/` にまとめ、`figure_and_table_index.csv` に各図・元表の場所を記録します。

In [12]:
# Updated C10: read saved C6B/C7/C9 results only.
from pathlib import Path
import importlib.util,sys,urllib.request,unittest
from IPython.display import display,Markdown

REPORT_COMMIT="5d1a50ff40e2ab6c85a7bb3385caa14495900e11"
OPPORTUNITY_DIR=None  # C6B output, e.g. /content/drive/MyDrive/卒論/research_v2_opportunity/<ID>
RUN_DIR_FOR_REPORT=None  # C7/C9 run folder, e.g. .../research_v2_runs/<ID>
if OPPORTUNITY_DIR is None:
    OPPORTUNITY_DIR=(globals().get("opportunity_result") or {}).get("output_dir")
if RUN_DIR_FOR_REPORT is None:
    RUN_DIR_FOR_REPORT=(globals().get("ctx") or {}).get("folder") or globals().get("RUN_DIR")
if OPPORTUNITY_DIR is None or RUN_DIR_FOR_REPORT is None:
    raise ValueError("OPPORTUNITY_DIRにC6B保存先、RUN_DIR_FOR_REPORTにC7/C9の実行フォルダを設定してください。")
report_code_root=Path("/content/mahjong_opportunity_report")/REPORT_COMMIT
report_code_root.mkdir(parents=True,exist_ok=True)
for name in ["opportunity_report.py","test_opportunity_report.py"]:
    url=f"https://raw.githubusercontent.com/yuukienomoto/report_-syobon_n/{REPORT_COMMIT}/research_v2/supplement/{name}"
    (report_code_root/name).write_bytes(urllib.request.urlopen(url,timeout=60).read())
spec=importlib.util.spec_from_file_location("opportunity_report",report_code_root/"opportunity_report.py")
report_module=importlib.util.module_from_spec(spec)
sys.modules[spec.name]=report_module
spec.loader.exec_module(report_module)
spec_test=importlib.util.spec_from_file_location("test_opportunity_report",report_code_root/"test_opportunity_report.py")
report_tests=importlib.util.module_from_spec(spec_test)
spec_test.loader.exec_module(report_tests)
checks=unittest.TextTestRunner(verbosity=1).run(unittest.defaultTestLoader.loadTestsFromModule(report_tests))
if not checks.wasSuccessful():raise RuntimeError("更新レポートの回帰テスト失敗。保存済み入力は変更せず停止します。")
updated_c10_result=report_module.updated_thesis_report(OPPORTUNITY_DIR,RUN_DIR_FOR_REPORT)
display(updated_c10_result["comparison"])
display(updated_c10_result["performance"])
display(Markdown(updated_c10_result["report_path"].read_text(encoding="utf-8")))
print("更新版C10の保存先:",updated_c10_result["output_dir"])


....
----------------------------------------------------------------------
Ran 4 tests in 1.397s

OK


,feature,japanese,opportunity_overlap_mean_diff_high_minus_low,opportunity_overlap_smd,opportunity_overlap_low_valid_n,opportunity_overlap_high_valid_n,traditional_overlap_mean_diff_high_minus_low,traditional_overlap_smd,traditional_overlap_low_valid_n,traditional_overlap_high_valid_n,...,traditional_disjoint_mean_diff_high_minus_low,traditional_disjoint_smd,traditional_disjoint_low_valid_n,traditional_disjoint_high_valid_n,aggregation_direction,overlap_disjoint_direction,disjoint_minus_overlap_mean_diff,disjoint_minus_overlap_smd,mean_diff_magnitude_ratio_disjoint_to_overlap,minimum_valid_n_opportunity
0,first_response_genbutsu_rate,成立リーチごとの初回応答現物率,-0.019544,-0.286068,2463,752,-0.020029,-0.259651,2355,740,...,-0.020692,-0.267486,410,114,同方向,同方向,0.000408,0.010359,0.979138,117
1,response_non_genbutsu_rate,対リーチ応答非現物率,0.005900,0.151145,2463,752,0.008395,0.176284,2355,740,...,0.009103,0.192646,410,114,同方向,同方向,0.001387,0.036700,1.235108,117
2,response_suji_nongenbutsu_rate,対リーチ応答非現物スジ率,0.004171,0.166931,2463,752,0.000603,0.020633,2355,740,...,0.002211,0.075190,410,114,同方向,同方向,0.000597,0.018452,1.143111,117
3,response_nonsuji_suited_rate,対リーチ応答非現物非スジ数牌率,0.003059,0.079140,2463,752,0.011603,0.244000,2355,740,...,0.009740,0.195791,410,114,同方向,同方向,-0.000453,-0.014224,0.851865,117
4,early_noten_non_genbutsu_rate,自己ツモ6回以下・打牌後形状ノーテン非現物率,0.008401,0.064856,2463,752,0.069503,0.663373,65,16,...,0.017714,0.133661,5,2,同方向,同方向,0.001051,0.006921,1.125055,117
5,mid_noten_non_genbutsu_rate,自己ツモ7～12回・打牌後形状ノーテン非現物率,0.009893,0.175749,2463,752,0.009858,0.147930,1724,557,...,0.006618,0.101206,295,84,同方向,同方向,-0.001287,-0.020802,0.869900,117
6,late_noten_non_genbutsu_rate,自己ツモ13回以上・打牌後形状ノーテン非現物率,-0.001158,-0.016739,2463,752,0.014656,0.209073,410,104,...,-0.009173,-0.133836,76,16,逆方向,逆方向,0.002058,0.030537,0.777162,117
7,roundstart_top_non_genbutsu_rate,局開始時単独1位の非現物応答率,0.006190,0.068435,2460,752,NaN,NaN,2,0,...,NaN,NaN,0,0,比較不能,同方向,0.004772,0.050435,1.771042,117
8,roundstart_middle_non_genbutsu_rate,局開始時同点なし2・3位の非現物応答率,0.008155,0.122823,2463,752,-0.004964,-0.065990,86,30,...,-0.046845,-0.462361,11,4,逆方向,同方向,-0.000531,-0.008676,0.934942,117
9,roundstart_last_non_genbutsu_rate,局開始時単独4位の非現物応答率,0.011265,0.105835,2437,747,NaN,NaN,0,0,...,NaN,NaN,0,0,比較不能,同方向,-0.000275,-0.002470,0.975548,117


,version,dataset,roc_auc,status,accuracy,balanced_accuracy,n,weak,strong,estimator,strong_ratio,f1_strong,f1_macro,recall_weak,recall_strong,tn,fp,fn,tp
0,legacy_random_reference,random_5fold,0.834000,historical_reference_different_split_possible_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,legacy_external_reference,external_old_preprocessing,NaN,historical_reference_different_population,0.355900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,legacy_H5_reference,fold1,NaN,user_reported_unavailable,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,legacy_H5_reference,fold2,0.580615,user_reported_not_rerun,NaN,0.506831,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,legacy_H5_reference,fold3,0.454563,user_reported_not_rerun,NaN,0.452368,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,legacy_H5_reference,external,0.374715,user_reported_not_rerun,NaN,0.404974,713.0,97.0,616.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,legacy23_same_protocol,fold1,NaN,saved_C7_C9_reused,0.877301,NaN,489.0,489.0,0.0,LightGBM,0.000000,0.000000,0.467320,0.877301,NaN,429.0,60.0,0.0,0.0
7,legacy23_same_protocol,fold1,NaN,saved_C7_C9_reused,1.000000,NaN,489.0,489.0,0.0,always_weak,0.000000,0.000000,0.500000,1.000000,NaN,489.0,0.0,0.0,0.0
8,legacy23_same_protocol,fold1,NaN,saved_C7_C9_reused,0.000000,NaN,489.0,489.0,0.0,always_strong,0.000000,0.000000,0.000000,0.000000,NaN,0.0,489.0,0.0,0.0
9,legacy23_same_protocol,fold1,NaN,saved_C7_C9_reused,1.000000,NaN,489.0,489.0,0.0,training_majority,0.000000,0.000000,0.500000,1.000000,NaN,489.0,0.0,0.0,0.0


# 卒論結果・考察 更新版：機会数方式を主分析候補とする

## 研究方法と集計の意味
しょぼんｎの段位16以下と18以上の時期を比較する。段位17は履歴に残し、終端の採点から除く。初期1500観測対局を含む時間帯までを採点外とし、hour_invariant窓を使用する。
主分析候補はC6Bのopportunity_overlap。各10対局窓内で分子と分母をそれぞれ合算して割り、その後、有効な窓を等重みで比較する。全イベントの単純合算とは推定対象が異なる。分母0は欠損とし、0補完しない。
18項目は機会あたりの割合。表ドラ枚数平均と待ち種類数平均の2項目は、合計数量÷合計イベント数による加重平均。従来の対局率10個の単純平均をtraditional_overlap感度分析として残す。
SMDは群内分散の自由度加重プール標準偏差で標準化した記述量。有効窓数・欠損率が方式ごとに異なるため、共通有効窓のCSVも併せて確認する。
主分析3215窓、非重複547窓、元対局最大再利用10回。

## 主分析候補：機会数方式の20特徴量
- 成立リーチごとの初回応答現物率（first_response_genbutsu_rate）：高−低=-0.019544、SMD=-0.286068、有効窓 低2463/高752。
- 対リーチ応答非現物率（response_non_genbutsu_rate）：高−低=+0.005900、SMD=+0.151145、有効窓 低2463/高752。
- 対リーチ応答非現物スジ率（response_suji_nongenbutsu_rate）：高−低=+0.004171、SMD=+0.166931、有効窓 低2463/高752。
- 対リーチ応答非現物非スジ数牌率（response_nonsuji_suited_rate）：高−低=+0.003059、SMD=+0.079140、有効窓 低2463/高752。
- 自己ツモ6回以下・打牌後形状ノーテン非現物率（early_noten_non_genbutsu_rate）：高−低=+0.008401、SMD=+0.064856、有効窓 低2463/高752。
- 自己ツモ7～12回・打牌後形状ノーテン非現物率（mid_noten_non_genbutsu_rate）：高−低=+0.009893、SMD=+0.175749、有効窓 低2463/高752。
- 自己ツモ13回以上・打牌後形状ノーテン非現物率（late_noten_non_genbutsu_rate）：高−低=-0.001158、SMD=-0.016739、有効窓 低2463/高752。
- 局開始時単独1位の非現物応答率（roundstart_top_non_genbutsu_rate）：高−低=+0.006190、SMD=+0.068435、有効窓 低2460/高752。
- 局開始時同点なし2・3位の非現物応答率（roundstart_middle_non_genbutsu_rate）：高−低=+0.008155、SMD=+0.122823、有効窓 低2463/高752。
- 局開始時単独4位の非現物応答率（roundstart_last_non_genbutsu_rate）：高−低=+0.011265、SMD=+0.105835、有効窓 低2437/高747。
- 成立リーチ時の表ドラ枚数平均（riichi_omote_dora_mean）：高−低=-0.003299、SMD=-0.019231、有効窓 低2463/高752。
- 自分の成立リーチに占める追っかけ割合（chasing_riichi_share）：高−低=+0.000194、SMD=+0.002185、有効窓 低2463/高752。
- 成立追っかけリーチの形状待ち種類数平均（chasing_wait_types_mean）：高−低=-0.013517、SMD=-0.036031、有効窓 低2302/高713。
- 成立追っかけリーチの両面解釈可能率（chasing_ryanmen_compatible_rate）：高−低=-0.008581、SMD=-0.028034、有効窓 低2302/高713。
- 成立追っかけリーチの表ドラ0割合（chasing_zero_omote_dora_rate）：高−低=+0.010141、SMD=+0.031506、有効窓 低2302/高713。
- 1回以上副露した局の割合（open_meld_round_rate）：高−低=+0.004974、SMD=+0.096098、有効窓 低2463/高752。
- 自己ツモ6回以下の初副露局割合（early_first_open_meld_rate）：高−低=+0.008910、SMD=+0.211162、有効窓 低2463/高752。
- 自己ツモ13回以上の初副露局割合（late_first_open_meld_rate）：高−低=-0.000484、SMD=-0.029844、有効窓 低2463/高752。
- 初副露成立直後に表ドラを持つ局割合（first_open_meld_with_omote_dora_rate）：高−低=+0.002875、SMD=+0.081454、有効窓 低2463/高752。
- 通常流局時の手牌開示率（exhaustive_draw_disclosed_tenpai_rate）：高−低=-0.015848、SMD=-0.120231、有効窓 低2463/高752。

## 感度分析：従来方式・非重複窓
方式間で差の符号が逆転した特徴量は late_noten_non_genbutsu_rate、roundstart_middle_non_genbutsu_rate。
以下の数値は方式を併記する。同方向は符号の一致だけを示す。大きさは平均差・SMDの変化を確認し、有効窓が少ない場合や片群欠損の場合は安定性を判断しない。
- 成立リーチごとの初回応答現物率：従来単純平均 高−低=-0.020029、SMD=-0.259651、有効窓 低2355/高740。集計方式間=同方向。機会数・非重複 高−低=-0.019136、SMD=-0.275710、有効窓 低430/高117、重複窓との方向=同方向。非重複−重複の平均差変化=+0.000408、SMD変化=+0.010359。
- 対リーチ応答非現物率：従来単純平均 高−低=+0.008395、SMD=+0.176284、有効窓 低2355/高740。集計方式間=同方向。機会数・非重複 高−低=+0.007287、SMD=+0.187845、有効窓 低430/高117、重複窓との方向=同方向。非重複−重複の平均差変化=+0.001387、SMD変化=+0.036700。
- 対リーチ応答非現物スジ率：従来単純平均 高−低=+0.000603、SMD=+0.020633、有効窓 低2355/高740。集計方式間=同方向。機会数・非重複 高−低=+0.004768、SMD=+0.185383、有効窓 低430/高117、重複窓との方向=同方向。非重複−重複の平均差変化=+0.000597、SMD変化=+0.018452。
- 対リーチ応答非現物非スジ数牌率：従来単純平均 高−低=+0.011603、SMD=+0.244000、有効窓 低2355/高740。集計方式間=同方向。機会数・非重複 高−低=+0.002606、SMD=+0.064916、有効窓 低430/高117、重複窓との方向=同方向。非重複−重複の平均差変化=-0.000453、SMD変化=-0.014224。
- 自己ツモ6回以下・打牌後形状ノーテン非現物率：従来単純平均 高−低=+0.069503、SMD=+0.663373、有効窓 低65/高16。集計方式間=同方向。機会数・非重複 高−低=+0.009452、SMD=+0.071777、有効窓 低430/高117、重複窓との方向=同方向。非重複−重複の平均差変化=+0.001051、SMD変化=+0.006921。
- 自己ツモ7～12回・打牌後形状ノーテン非現物率：従来単純平均 高−低=+0.009858、SMD=+0.147930、有効窓 低1724/高557。集計方式間=同方向。機会数・非重複 高−低=+0.008606、SMD=+0.154947、有効窓 低430/高117、重複窓との方向=同方向。非重複−重複の平均差変化=-0.001287、SMD変化=-0.020802。
- 自己ツモ13回以上・打牌後形状ノーテン非現物率：従来単純平均 高−低=+0.014656、SMD=+0.209073、有効窓 低410/高104。集計方式間=逆方向。機会数・非重複 高−低=+0.000900、SMD=+0.013798、有効窓 低430/高117、重複窓との方向=逆方向。非重複−重複の平均差変化=+0.002058、SMD変化=+0.030537。
- 局開始時単独1位の非現物応答率：従来単純平均 高−低=算出不可、SMD=算出不可、有効窓 低2/高0。集計方式間=比較不能。機会数・非重複 高−低=+0.010962、SMD=+0.118871、有効窓 低429/高117、重複窓との方向=同方向。非重複−重複の平均差変化=+0.004772、SMD変化=+0.050435。
- 局開始時同点なし2・3位の非現物応答率：従来単純平均 高−低=-0.004964、SMD=-0.065990、有効窓 低86/高30。集計方式間=逆方向。機会数・非重複 高−低=+0.007625、SMD=+0.114146、有効窓 低430/高117、重複窓との方向=同方向。非重複−重複の平均差変化=-0.000531、SMD変化=-0.008676。
- 局開始時単独4位の非現物応答率：従来単純平均 高−低=算出不可、SMD=算出不可、有効窓 低0/高0。集計方式間=比較不能。機会数・非重複 高−低=+0.010989、SMD=+0.103365、有効窓 低426/高117、重複窓との方向=同方向。非重複−重複の平均差変化=-0.000275、SMD変化=-0.002470。
- 成立リーチ時の表ドラ枚数平均：従来単純平均 高−低=-0.000925、SMD=-0.005316、有効窓 低551/高218。集計方式間=同方向。機会数・非重複 高−低=-0.022531、SMD=-0.132197、有効窓 低430/高117、重複窓との方向=同方向。非重複−重複の平均差変化=-0.019232、SMD変化=-0.112966。
- 自分の成立リーチに占める追っかけ割合：従来単純平均 高−低=+0.012106、SMD=+0.137012、有効窓 低551/高218。集計方式間=同方向。機会数・非重複 高−低=+0.000260、SMD=+0.002873、有効窓 低430/高117、重複窓との方向=同方向。非重複−重複の平均差変化=+0.000066、SMD変化=+0.000689。
- 成立追っかけリーチの形状待ち種類数平均：従来単純平均 高−低=算出不可、SMD=算出不可、有効窓 低0/高0。集計方式間=比較不能。機会数・非重複 高−低=-0.036929、SMD=-0.099584、有効窓 低402/高110、重複窓との方向=同方向。非重複−重複の平均差変化=-0.023412、SMD変化=-0.063553。
- 成立追っかけリーチの両面解釈可能率：従来単純平均 高−低=算出不可、SMD=算出不可、有効窓 低0/高0。集計方式間=比較不能。機会数・非重複 高−低=-0.022578、SMD=-0.072031、有効窓 低402/高110、重複窓との方向=同方向。非重複−重複の平均差変化=-0.013998、SMD変化=-0.043998。
- 成立追っかけリーチの表ドラ0割合：従来単純平均 高−低=算出不可、SMD=算出不可、有効窓 低0/高0。集計方式間=比較不能。機会数・非重複 高−低=+0.047042、SMD=+0.141307、有効窓 低402/高110、重複窓との方向=同方向。非重複−重複の平均差変化=+0.036901、SMD変化=+0.109801。
- 1回以上副露した局の割合：従来単純平均 高−低=+0.005761、SMD=+0.107459、有効窓 低2463/高752。集計方式間=同方向。機会数・非重複 高−低=+0.001545、SMD=+0.029455、有効窓 低430/高117、重複窓との方向=同方向。非重複−重複の平均差変化=-0.003429、SMD変化=-0.066643。
- 自己ツモ6回以下の初副露局割合：従来単純平均 高−低=+0.009561、SMD=+0.217897、有効窓 低2463/高752。集計方式間=同方向。機会数・非重複 高−低=+0.005206、SMD=+0.120817、有効窓 低430/高117、重複窓との方向=同方向。非重複−重複の平均差変化=-0.003704、SMD変化=-0.090345。
- 自己ツモ13回以上の初副露局割合：従来単純平均 高−低=-0.001290、SMD=-0.077672、有効窓 低2463/高752。集計方式間=同方向。機会数・非重複 高−低=-0.000548、SMD=-0.033639、有効窓 低430/高117、重複窓との方向=同方向。非重複−重複の平均差変化=-0.000065、SMD変化=-0.003794。
- 初副露成立直後に表ドラを持つ局割合：従来単純平均 高−低=+0.003632、SMD=+0.098911、有効窓 低2463/高752。集計方式間=同方向。機会数・非重複 高−低=+0.000568、SMD=+0.016225、有効窓 低430/高117、重複窓との方向=同方向。非重複−重複の平均差変化=-0.002306、SMD変化=-0.065228。
- 通常流局時の手牌開示率：従来単純平均 高−低=-0.029130、SMD=-0.232067、有効窓 低267/高72。集計方式間=同方向。機会数・非重複 高−低=-0.008923、SMD=-0.066761、有効窓 低430/高117、重複窓との方向=同方向。非重複−重複の平均差変化=+0.006925、SMD変化=+0.053471。

## 補助分析：保存済みC7・C9の分類結果
C7・C9の既存CSVを読み込んで掲載した。モデル入力は従来の訓練区間のみで補完した対局特徴量の10対局平均。機会数方式を適用した対象は記述分析である。分類性能を新しい集計方式の評価値へ読み替えない。
- legacy23_same_protocol/fold1/LightGBM：AUC=算出不可、Balanced Accuracy=算出不可、Accuracy=+0.877301、F1強=+0.000000。
- legacy23_same_protocol/fold1/always_weak：AUC=算出不可、Balanced Accuracy=算出不可、Accuracy=+1.000000、F1強=+0.000000。
- legacy23_same_protocol/fold1/always_strong：AUC=算出不可、Balanced Accuracy=算出不可、Accuracy=+0.000000、F1強=+0.000000。
- legacy23_same_protocol/fold1/training_majority：AUC=算出不可、Balanced Accuracy=算出不可、Accuracy=+1.000000、F1強=+0.000000。
- legacy23_same_protocol/fold2/LightGBM：AUC=+0.580615、Balanced Accuracy=+0.506831、Accuracy=+0.590437、F1強=+0.182573。
- legacy23_same_protocol/fold2/always_weak：AUC=+0.500000、Balanced Accuracy=+0.500000、Accuracy=+0.607069、F1強=+0.000000。
- legacy23_same_protocol/fold2/always_strong：AUC=+0.500000、Balanced Accuracy=+0.500000、Accuracy=+0.392931、F1強=+0.564179。
- legacy23_same_protocol/fold2/training_majority：AUC=+0.500000、Balanced Accuracy=+0.500000、Accuracy=+0.607069、F1強=+0.000000。
- legacy23_same_protocol/fold3/LightGBM：AUC=+0.454563、Balanced Accuracy=+0.452368、Accuracy=+0.529652、F1強=+0.240924。
- legacy23_same_protocol/fold3/always_weak：AUC=+0.500000、Balanced Accuracy=+0.500000、Accuracy=+0.671779、F1強=+0.000000。
- legacy23_same_protocol/fold3/always_strong：AUC=+0.500000、Balanced Accuracy=+0.500000、Accuracy=+0.328221、F1強=+0.494226。
- legacy23_same_protocol/fold3/training_majority：AUC=+0.500000、Balanced Accuracy=+0.500000、Accuracy=+0.671779、F1強=+0.000000。
- v2_fixed_same_protocol/fold1/LightGBM：AUC=算出不可、Balanced Accuracy=算出不可、Accuracy=+0.905930、F1強=+0.000000。
- v2_fixed_same_protocol/fold1/always_weak：AUC=算出不可、Balanced Accuracy=算出不可、Accuracy=+1.000000、F1強=+0.000000。
- v2_fixed_same_protocol/fold1/always_strong：AUC=算出不可、Balanced Accuracy=算出不可、Accuracy=+0.000000、F1強=+0.000000。
- v2_fixed_same_protocol/fold1/training_majority：AUC=算出不可、Balanced Accuracy=算出不可、Accuracy=+1.000000、F1強=+0.000000。
- v2_fixed_same_protocol/fold2/LightGBM：AUC=+0.561372、Balanced Accuracy=+0.520222、Accuracy=+0.611227、F1強=+0.161435。
- v2_fixed_same_protocol/fold2/always_weak：AUC=+0.500000、Balanced Accuracy=+0.500000、Accuracy=+0.607069、F1強=+0.000000。
- v2_fixed_same_protocol/fold2/always_strong：AUC=+0.500000、Balanced Accuracy=+0.500000、Accuracy=+0.392931、F1強=+0.564179。
- v2_fixed_same_protocol/fold2/training_majority：AUC=+0.500000、Balanced Accuracy=+0.500000、Accuracy=+0.607069、F1強=+0.000000。
- v2_fixed_same_protocol/fold3/LightGBM：AUC=+0.424136、Balanced Accuracy=+0.414615、Accuracy=+0.492843、F1強=+0.194805。
- v2_fixed_same_protocol/fold3/always_weak：AUC=+0.500000、Balanced Accuracy=+0.500000、Accuracy=+0.671779、F1強=+0.000000。
- v2_fixed_same_protocol/fold3/always_strong：AUC=+0.500000、Balanced Accuracy=+0.500000、Accuracy=+0.328221、F1強=+0.494226。
- v2_fixed_same_protocol/fold3/training_majority：AUC=+0.500000、Balanced Accuracy=+0.500000、Accuracy=+0.671779、F1強=+0.000000。
- v2_fixed_external/external/LightGBM：AUC=+0.292743、Balanced Accuracy=+0.354524、Accuracy=+0.274895、F1強=+0.368742。
- v2_fixed_external/external/always_weak：AUC=+0.500000、Balanced Accuracy=+0.500000、Accuracy=+0.136045、F1強=+0.000000。
- v2_fixed_external/external/always_strong：AUC=+0.500000、Balanced Accuracy=+0.500000、Accuracy=+0.863955、F1強=+0.927013。
- v2_fixed_external/external/training_majority：AUC=+0.500000、Balanced Accuracy=+0.500000、Accuracy=+0.136045、F1強=+0.000000。

## 考察と限界
機会数方式は、機会のない対局を含む窓の行動情報を利用できる。方式間で符号や大きさが変わる場合、機会数による重みの変更と有効窓の構成変化が影響した可能性を分けて検討する。
非重複窓でも同じ人物の時系列相関は残る。重複窓を独立標本として検定・信頼区間を算出していない。
order_verified=False、full_history_confirmed=False。hour_invariantは観測履歴内の順序不変性に限られる。段位と時間経過、卓条件、対戦相手、行動機会が交絡する。窓終端と履歴の段位が混在する場合もある。
同一人物の時期別比較として解釈し、実力向上の原因や一般の強者・弱者全体の差を断定しない。外部評価結果を用いたモデル・特徴量・閾値の調整は行わない。

更新版C10の保存先: /content/drive/MyDrive/卒論/research_v2_runs/20261105T071722Z_5da6c9ad/final_report_opportunity/20261105T074007Z_dc9e3e18
